In [26]:
import sys
sys.path.append("/rds/general/user/xy823/home/env0418/ai_clinician_yxr/py_ai_clinician-master")
import os

# Get the current working directory
current_directory = os.getcwd()

# Print out the current directory
print("Current directory:", current_directory)

Current directory: /rds/general/user/xy823/home/env0418/ai_clinician_yxr/py_ai_clinician-master/continuous/0614/Fed


In [27]:
# Get the current working directory
current_directory = os.getcwd()

# Print out the current directory
print("Current directory:", current_directory)

Current directory: /rds/general/user/xy823/home/env0418/ai_clinician_yxr/py_ai_clinician-master/continuous/0614/Fed


In [28]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [29]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
import numpy as np
import math
import os
import random
import numpy as np
import pandas as pd
from pandas import DataFrame
import pickle
import math
import copy

TensorFlow version: 1.15.0


In [30]:
state_features = [
    "Albumin",
    "Arterial_BE",
    "Arterial_lactate",
    "Arterial_pH",
    "BUN",
    "CO2_mEqL",
    "Calcium",
    "Chloride",
    "Creatinine",
    "DiaBP",
    "FiO2_1",
    "GCS",
    "Glucose",
    "HCO3",
    "HR",
    "Hb",
    "INR",
    "Ionised_Ca",
    "Magnesium",
    "MeanBP",
    "PT",
    "PTT",
    "PaO2_FiO2",
    "Platelets_count",
    "Potassium",
    "RR",
    "SGOT",
    "SGPT",
    "SIRS",
    "SOFA",
    "Shock_Index",
    "Sodium",
    "SpO2",
    "SysBP",
    "Temp_C",
    "Total_bili",
    "WBC_count",
    "Weight_kg",
    "age",
    "elixhauser",
    "gender",
    "mechvent",
    "output_4hourly",
    "output_total",
    "paCO2",
    "paO2",
    "re_admission",
    "bloc"
]
DQN_features=state_features

In [31]:
# generates batches for the Q network - depending on train and eval_type, can select data from train/val/test sets.
def process_batch(node,size, train=True, eval_type = None,clip_reward=True):
#     print("---process_batch w:",eval_type)
    if not train:# if train = false,have not trained yet
        if eval_type is None:
            raise Exception('Provide eval_type to process_batch')
        elif eval_type == 'train':
            a = node.df.copy()
        elif eval_type == 'val':
            a = node.val_df.copy()
        elif eval_type == 'test':
            a = node.test_df.copy()
        
        ##############
        elif eval_type == 'fed':# use training set for now?????
            print("---process_batch w:",eval_type)
            a = node.test_df.copy() ###no, should use test set###
        else:
            raise Exception('Unknown eval_type')
    else:
        if per_flag:
            # uses prioritised exp replay -PER
#             print("----")
#             data=node.df
#             print(data)
            a = node.df.sample(n=size, weights=node.df['prob'])
        else:
            a = node.df.sample(n=size)
    states = None
    actions = None
    rewards = None
    next_states = None
    done_flags = None
    for i in a.index:
        cur_state = a.loc[i,state_features]
        iv = int(a.loc[i, 'iv_input'])
        vaso = int(a.loc[i, 'vaso_input'])
        action = action_map[iv,vaso]
        reward = a.loc[i,'reward']
        
        ### add 0612:
        if clip_reward:
            if reward > 1: reward = 1
            if reward < -1: reward = -1

        if i != node.df.index[-1]:
            # if not terminal step in trajectory             
            if node.df.loc[i, 'icustayid'] == node.df.loc[i+1, 'icustayid']:
                next_state = node.df.loc[i + 1, state_features]
                done = 0
            else:
                # trajectory is finished
                next_state = np.zeros(len(cur_state))
                done = 1
        else:
            # last entry in df is the final state of that trajectory
            next_state = np.zeros(len(cur_state))
            done = 1

        if states is None:
            states = copy.deepcopy(cur_state)
        else:
            states = np.vstack((states,cur_state))

        if actions is None:
            actions = [action]
        else:
            actions = np.vstack((actions,action))

        if rewards is None:
            rewards = [reward]
        else:
            rewards = np.vstack((rewards,reward))

        if next_states is None:
            next_states = copy.deepcopy(next_state)
        else:
            next_states = np.vstack((next_states,next_state))

        if done_flags is None:
            done_flags = [done]
        else:
            done_flags = np.vstack((done_flags,done))
    
    return (states, np.squeeze(actions), np.squeeze(rewards), next_states, np.squeeze(done_flags), a)

In [32]:
# function is needed to update parameters between main and target network
# tf_vars are the trainable variables to update, and tau is the rate at which to update
# returns tf ops corresponding to the updates
# tf_vars：可训练的变量列表，包括主网络和目标网络的所有参数。
# tau：更新速率，决定了参数在每次更新时受到主网络参数和目标网络参数的影响程度
def update_target_graph(tf_vars,tau):
    total_vars = len(tf_vars)
    op_holder = []
    for idx,var in enumerate(tf_vars[0:int(total_vars/2)]):
        op_holder.append(tf_vars[idx+int(total_vars/2)].assign((var.value()*tau) + ((1-tau)*tf_vars[idx+int(total_vars/2)].value())))
    return op_holder

In [33]:
def update_target(op_holder,sess):
    for op in op_holder:
        sess.run(op)

In [34]:
# hidden_1_size = 400
# hidden_2_size = 400
hidden_1_size = 128
hidden_2_size = 128
# Q-network uses Leaky ReLU activation
class Qnetwork():
    def __init__(self, graph, scope):
        self.phase = tf.placeholder(tf.bool)
        self.num_actions = 25
        self.input_size = len(state_features)## 200

        self.graph = graph
        with self.graph.as_default():
            with tf.variable_scope(scope, reuse=tf.AUTO_REUSE):
                # 使用 tf.variable_scope 和 reuse=tf.AUTO_REUSE 确保变量重用
                self.state = tf.placeholder(tf.float32, shape=[None, self.input_size], name="input_state")
                self.fc_1 = tf.contrib.layers.fully_connected(self.state, hidden_1_size, activation_fn=None)
                self.fc_1_bn = tf.contrib.layers.batch_norm(self.fc_1, center=True, scale=True, is_training=self.phase)
                self.fc_1_ac = tf.maximum(self.fc_1_bn, self.fc_1_bn * 0.5)  # Leaky ReLU activation
                self.fc_2 = tf.contrib.layers.fully_connected(self.fc_1_ac, hidden_2_size, activation_fn=None)
                self.fc_2_bn = tf.contrib.layers.batch_norm(self.fc_2, center=True, scale=True, is_training=self.phase)
                self.fc_2_ac = tf.maximum(self.fc_2_bn, self.fc_2_bn * 0.5)

                self.streamA, self.streamV = tf.split(self.fc_2_ac, 2, axis=1)
                self.AW = tf.Variable(tf.random_normal([hidden_2_size // 2, self.num_actions]))
                self.VW = tf.Variable(tf.random_normal([hidden_2_size // 2, 1]))
                self.Advantage = tf.matmul(self.streamA, self.AW)
                self.Value = tf.matmul(self.streamV, self.VW)

                self.q_output = self.Value + tf.subtract(self.Advantage, tf.reduce_mean(self.Advantage, axis=1, keepdims=True))
                self.predict = tf.argmax(self.q_output, 1, name='predict')

                self.targetQ = tf.placeholder(shape=[None], dtype=tf.float32)
                self.actions = tf.placeholder(shape=[None], dtype=tf.int32)
                self.actions_onehot = tf.one_hot(self.actions, self.num_actions, dtype=tf.float32)
                self.imp_weights = tf.placeholder(shape=[None], dtype=tf.float32)
                self.Q = tf.reduce_sum(tf.multiply(self.q_output, self.actions_onehot), axis=1)
                self.reg_vector = tf.maximum(tf.abs(self.Q) - REWARD_THRESHOLD, 0)
                self.reg_term = tf.reduce_sum(self.reg_vector)
                self.abs_error = tf.abs(self.targetQ - self.Q)
                self.td_error = tf.square(self.targetQ - self.Q)
                self.old_loss = tf.reduce_mean(self.td_error)
                self.per_error = tf.multiply(self.td_error, self.imp_weights)
                # loss function
                if per_flag:# prioritized experience replay 
                    self.loss = tf.reduce_mean(self.per_error) + reg_lambda * self.reg_term
                else:
                    self.loss = self.old_loss + reg_lambda * self.reg_term

                self.trainer = tf.train.AdamOptimizer(learning_rate=0.0002)
                self.update_ops = tf.get_collection(tf.GraphKeys.UPDATE_OPS, scope=scope)
                with tf.control_dependencies(self.update_ops):
                    self.update_model = self.trainer.minimize(self.loss)

            
#     def get_weights(self, sess):
#         variables = tf.trainable_variables()
#         return sess.run(variables)

#     def set_weights(self, sess, weights):
#         variables = tf.trainable_variables()
#         assign_ops = [var.assign(weights[idx]) for idx, var in enumerate(variables)]
#         sess.run(assign_ops)
    def get_weights(self, sess):
        variables = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=tf.get_variable_scope().name)
        return sess.run(variables)

    def set_weights(self, sess, weights):
        variables = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=tf.get_variable_scope().name)
        assign_ops = [var.assign(w) for var, w in zip(variables, weights)]
        sess.run(assign_ops)
        


In [35]:
#  Used to run diagnostics on the train set
phys_q_train = []
agent_q_train = []
phys_actions_tr = []
agent_actions_tr = []
def train_set_performance():
    count = 0
    global phys_q_train
    global agent_q_train
    global phys_actions
    global agent_actions
    phys_q_train = []
    agent_q_train = []
    phys_actions_tr = []
    agent_actions_tr = []
    for r in df.index:
        cur_state = [df.loc[r,state_features]]
        iv = int(df.loc[r, 'iv_input'])
        vaso = int(df.loc[r, 'vaso_input'])
        action = action_map[iv,vaso]
        output_q = np.squeeze(sess.run(mainQN.q_output, feed_dict = {mainQN.state : cur_state, mainQN.phase : False}))
        phys_q_train.append(output_q[action])
        agent_q_train.append(max(output_q))
        agent_actions_tr.append(np.argmax(output_q))
        phys_actions_tr.append(action)
        count += 1

In [36]:
def do_eval_new(sess,node,eval_type,server=None):# eval_type is train test val #### eval_type + fed_local
    print("---do_eval:", eval_type)
    states,actions,rewards,next_states, done_flags, _ = process_batch(node,size=1024,train=False,eval_type=eval_type) #size=None //0612
    print("states shape:", states.shape)
    print("actions shape:", actions.shape)
    print("rewards shape:", rewards.shape)
    print("next_states shape:", next_states.shape)
    print("done_flags shape:", done_flags.shape)
    # firstly get the chosen actions at the next timestep
    if eval_type=='fed':
        print(' start to evaluate fed policy locally...')
        with server.graph.as_default(): 
                
                # 使用主网络（mainQN）预测下一个状态的动作 actions_from_q1
                actions_from_q1 = sess.run(server.mainQN.predict,feed_dict={server.mainQN.state:next_states, server.mainQN.phase : 0})
                # Q values for the next timestep from target network, as part of the Double DQN update
                # the target network's output
                Q2 = sess.run(server.targetQN.q_output,feed_dict={server.targetQN.state:next_states, server.targetQN.phase : 0})

    else:         
        # 使用主网络（mainQN）预测下一个状态的动作 actions_from_q1
        actions_from_q1 = sess.run(node.mainQN.predict,feed_dict={node.mainQN.state:next_states, node.mainQN.phase : 0})
        # Q values for the next timestep from target network, as part of the Double DQN update
        # the target network's output
        Q2 = sess.run(node.targetQN.q_output,feed_dict={node.targetQN.state:next_states, node.targetQN.phase : 0})
    # handles the case when a trajectory is finished
    end_multiplier = 1 - done_flags
    # target Q value using Q values from target, and actions from main
    # range(batch_size) ---S' # #How many experiences to use for each training step.# 30
    # actions_from_q1-- argmax_a Q1 
    print("----do_eval----")
    print("Q2.shape:",Q2.shape)
    print("actions_from_q1:",actions_from_q1.shape)
    print(range(batch_size))
#     double_q_value = Q2[range(batch_size),actions_from_q1]
    temp=Q2.shape[0]
    print(range(temp))
    double_q_value = Q2[range(temp),actions_from_q1]

    # definition of target Q
    targetQ = rewards + (gamma*double_q_value * end_multiplier)

    # get the output q's, actions, and loss
    if eval_type=='fed':
        print(' evaluate fed policy locally...')
        with server.graph.as_default():  # new comment: 确保使用服务器的图
                q_output,actions_taken, abs_error = sess.run([server.mainQN.q_output,server.mainQN.predict, server.mainQN.abs_error], \
                    feed_dict={server.mainQN.state:states,
                               server.mainQN.targetQ:targetQ, 
                               server.mainQN.actions:actions,
                           server.mainQN.phase:False})
    else:
        q_output,actions_taken, abs_error = sess.run([node.mainQN.q_output,node.mainQN.predict, node.mainQN.abs_error], \
            feed_dict={node.mainQN.state:states,
                       node.mainQN.targetQ:targetQ, 
                       node.mainQN.actions:actions,
                       node.mainQN.phase:False})
    
    # return the relevant q values and actions
    # 输入是状态，输出是 Q 值
    print("q_output.shape:",q_output.shape)
    print("actions_taken.shape:",actions_taken.shape)
    phys_q = q_output[range(len(q_output)), actions]
    agent_q = q_output[range(len(q_output)), actions_taken]
    error = np.mean(abs_error)
    
    return phys_q, actions, agent_q, actions_taken, error

In [37]:
class NODE:
    def __init__(self, MIMICtable,name,graph,scope):
        print(MIMICtable.shape)
        self.table = MIMICtable.copy()
        self.name = name
        self.icustayidlist=self.table['icustayid']
        self.icuuniqueids=np.unique(self.icustayidlist)
        self.len_traj_all_patients=self.table.shape[0]
        
        self.train=[]
        self.val=[]
        self.test=[]
        
        self.train_auto=[]
        self.val_auto=[]
        self.test_auto=[]
        

         # Initialize Q-networks
        self.input_size = len(state_features)  # Assuming a fixed input size
        self.num_actions = 25  # Assuming a fixed number of actions

        self.graph = graph
        with self.graph.as_default():
            self.mainQN = Qnetwork(self.graph, scope=f'{scope}_main')
            self.targetQN = Qnetwork(self.graph, scope=f'{scope}_target')

        # Placeholder for model statistics
    
        self.model_stats = {
            'losses': [],
            'mean_targetQ': [],
            'errors': [],
            'agent_q':[],
            'phys_q':[],
            'weights': {}
        }
        
        self.df=None#train
        self.val_df=None
        self.tets_df=None
        
        self.pol_val = 0
        self.server_val = 0
        self.decay = 1
        self.decay_rate = 0.995 #decay_rate
        
    def show_size(self):
        print("Sizes of the lists:")
        print("train:", len(self.train))
        print("val:", len(self.val))
        print("test:", len(self.test))
        
        print("train_auto:", len(self.train_auto))
        print("val_auto:", len(self.val_auto))
        print("test_auto:", len(self.test_auto))
        
        print("\nShapes of the DataFrames:")
        if self.df is not None:
            print("df:", self.df.shape)
        else:
            print("df: None")
            
        if self.val_df is not None:
            print("val_df:", self.val_df.shape)
        else:
            print("val_df: None")
        
        if self.test_df is not None:
            print("test_df:", self.test_df.shape)
        else:
            print("test_df: None")


        
    def split_train_val_test(self,path):
        MIMICtable=self.table.copy()
        unique_icustayids = MIMICtable['icustayid'].unique()

        # 计算每个集合的大小
        train_size = int(len(unique_icustayids) * 0.7)
        val_size = int(len(unique_icustayids) * 0.1)
        test_size = len(unique_icustayids) - train_size - val_size

        # 随机选择相应比例的 icustayid 值
        train_icustayids = np.random.choice(unique_icustayids, size=train_size, replace=False)
        remaining_icustayids = np.setdiff1d(unique_icustayids, train_icustayids)
        val_icustayids = np.random.choice(remaining_icustayids, size=val_size, replace=False)
        test_icustayids = np.setdiff1d(remaining_icustayids, val_icustayids)

        # 根据划分结果提取相应的数据
        train_data = MIMICtable[MIMICtable['icustayid'].isin(train_icustayids)]
        val_data = MIMICtable[MIMICtable['icustayid'].isin(val_icustayids)]
        test_data = MIMICtable[MIMICtable['icustayid'].isin(test_icustayids)]

            # 打印 train_data 的行数和唯一的 icustayid 数量
        print("dataset num of rows:total- train-val-test:", len(MIMICtable),len(train_data),len(val_data),len(test_data))
        print("dataset num of unique_id: total-train-val-test:", len(MIMICtable['icustayid'].unique()),len(train_data['icustayid'].unique()),len(val_data['icustayid'].unique()),len(test_data['icustayid'].unique()))
        
        self.train=train_data
        self.val=val_data
        self.test=test_data
        
        # Print dataset information
        print("Number of unique icustayids:", len(unique_icustayids))
        print("Train size:", train_size)
        print("Validation size:", val_size)
        print("Test size:", test_size)
        
        print("Size of train_data:", len(train_data))
        print("Size of val_data:", len(val_data))
        print("Size of test_data:", len(test_data))
        
        print("Number of unique icustayids in train_data:", len(train_data['icustayid'].unique()))
        print("Number of unique icustayids in val_data:", len(val_data['icustayid'].unique()))
        print("Number of unique icustayids in test_data:", len(test_data['icustayid'].unique()))
        
        # Construct file paths
        train_path = f"{path}/{self.name}_train.csv"
        val_path = f"{path}/{self.name}_val.csv"
        test_path = f"{path}/{self.name}_test.csv"

        # Write the data to CSV files
        train_data.to_csv(train_path, index=False)
        val_data.to_csv(val_path, index=False)
        test_data.to_csv(test_path, index=False)


        
        


In [38]:
# train--rl_train_data_final_auto#### 204 columns
# train_org--train.csv===df_train.copy()===process===d_scale
# train['died_in_hosp'] = train_orig['died_in_hosp'] #### 205 columns

In [39]:
# orig -- split Nodes--Split train_test_val---Use auto_encoder

In [40]:
def random_split(idx, num_splits):
    
    split=[]
    random.seed(42)
    random.shuffle(idx)  # 将idx列表中的元素随机打乱
#     print(idx)
    split_size = len(idx) // num_splits  # 计算每份的大小
#     print(split_size)
    for i in range(num_splits-1):
        i=i*split_size
        # print(i)
        split.append(idx[i:i+split_size])
        # print(split)

    split.append(idx[(split_size)*(num_splits-1):len(idx)])
#     print(split)
    return split

In [41]:
import pandas as pd

# Function to read 10% of a CSV file
def read_10_percent(file_path):
    # Calculate the number of rows to skip
    total_rows = sum(1 for row in open(file_path)) - 1
    skip_rows = int(total_rows * 0.9)
    # Read the CSV file
    return pd.read_csv(file_path, skiprows=skip_rows, nrows=int(total_rows * 0.1))




In [42]:
def create_nodes(orig_path, path, seed=42, numNodes=3, Existed_files=True, per_flag=True):
    nodes_table = []
    for i in range(numNodes):
        node_name=f'node{i+1}'
        current_table=pd.read_csv(f"{path}/{node_name}_data_seed42.csv")
        nodes_table.append(current_table)
    
    nodes = []
    beta_start = 0.9
    for i in range(numNodes):
        graph = tf.Graph()
        # 为每个节点创建独立的图
        with graph.as_default():
            node_name = f"node{i+1}"
            node = NODE(nodes_table[i], node_name, graph, scope=f'node{i}')
            if Existed_files:
                node.train = pd.read_csv(f"{path}/{node_name}_rl_train_data_final_cont.csv")
                node.val = pd.read_csv(f"{path}/{node_name}_rl_val_data_final_cont.csv")
                node.test = pd.read_csv(f"{path}/{node_name}_rl_test_data_final_cont.csv")
                node.train_auto = copy.deepcopy(node.train)
                node.val_auto = copy.deepcopy(node.val)
                node.test_auto =  copy.deepcopy(node.test)
            else:
                print("Existed_files does not exist")
                node.split_train_val_test(path)
#                 node.Encoder(path)
#             node.train_auto['died_in_hosp'] = node.train['died_in_hosp']
#             node.val_auto['died_in_hosp'] = node.val['died_in_hosp']
#             node.test_auto['died_in_hosp'] = node.test['died_in_hosp']

            node.df = node.train_auto.copy()
            node.df['prob'] = abs(node.df['reward'])
            temp = 1.0 / node.df['prob']
            node.df['imp_weight'] = pow((1.0 / len(node.df) * temp), beta_start) 
            node.val_df = node.val_auto.copy()
            node.test_df = node.test_auto.copy()

            nodes.append(node)

    return nodes


In [43]:
# def create_nodes(orig_path, path, seed=42, numNodes=3, Existed_files=True, per_flag=True):
#     orig = pd.read_csv(orig_path)
#     print("Shape of the DataFrame 'orig':", orig.shape)
#     icustayidlist = orig['icustayid']
#     icuuniqueids = np.unique(icustayidlist)
#     random.seed(seed)
#     splits = random_split(icuuniqueids, numNodes)
#     nodes_table = []
#     for i, split in enumerate(splits):
#         mask = np.isin(icustayidlist, split)
#         current_node = orig.loc[mask, :]
#         nodes_table.append(current_node)

#     smallest_node = min(nodes_table, key=lambda x: x.shape[0])
#     print(f"Smallest node size (rows, columns): {smallest_node.shape}")
    
#     nodes = []
#     beta_start = 0.9
#     for i in range(numNodes):
#         graph = tf.Graph()
#         # 为每个节点创建独立的图
#         with graph.as_default():
#             node_name = f"N{i+1}"
#             node = NODE(nodes_table[i], node_name, graph, scope=f'node{i}')
#             if Existed_files:
#                 node.train = pd.read_csv(f"{path}/{node_name}_train.csv")
#                 node.val = pd.read_csv(f"{path}/{node_name}_val.csv")
#                 node.test = pd.read_csv(f"{path}/{node_name}_test.csv")
#                 node.train_auto = copy.deepcopy(node.train)
#                 node.val_auto = copy.deepcopy(node.val)
#                 node.test_auto =  copy.deepcopy(node.test)
#             else:
#                 print("Existed_files does not exist")
#                 node.split_train_val_test(path)
# #                 node.Encoder(path)
# #             node.train_auto['died_in_hosp'] = node.train['died_in_hosp']
# #             node.val_auto['died_in_hosp'] = node.val['died_in_hosp']
# #             node.test_auto['died_in_hosp'] = node.test['died_in_hosp']

#             node.df = node.train_auto.copy()
#             node.df['prob'] = abs(node.df['reward'])
#             temp = 1.0 / node.df['prob']
#             node.df['imp_weight'] = pow((1.0 / len(node.df) * temp), beta_start) 
#             node.val_df = node.val_auto.copy()
#             node.test_df = node.test_auto.copy()

#             nodes.append(node)

#     return nodes


In [44]:
def save_every(sess,node,round):
    _, _, agent_q_val, agent_actions_val, _ = do_eval_new(sess,node,eval_type = 'val')        
    _, _, agent_q_test, agent_actions_test, _ = do_eval_new(sess,node,eval_type = 'test')  
    _, _, agent_q_train, agent_actions_train, _ = do_eval_new(sess,node,eval_type = 'train')
    save_dir = f"FedDQN_64_2k/dqn_auto_{node.name}_{round}"
    with open(save_dir + f'{node.name}_R{round}_dqn_normal_actions_train.p', 'wb') as f:
        pickle.dump(agent_actions_train, f)
    with open(save_dir + f'{node.name}_R{round}_dqn_normal_actions_val.p', 'wb') as f:
        pickle.dump(agent_actions_val, f)
    with open(save_dir + f'{node.name}_R{round}_dqn_normal_actions_test.p', 'wb') as f:
        pickle.dump(agent_actions_test, f)
    with open(save_dir + f'{node.name}_R{round}_dqn_normal_q_train.p', 'wb') as f:
        pickle.dump(agent_q_train, f)
    with open(save_dir + f'{node.name}_R{round}_dqn_normal_q_val.p', 'wb') as f:
        pickle.dump(agent_q_val, f)
    with open(save_dir + f'{node.name}_R{round}_dqn_normal_q_test.p', 'wb') as f:
        pickle.dump(agent_q_test, f)

In [45]:

class Server:
    def __init__(self, input_size,num_actions,local_epochs, global_iters,graph,scope,temp_a=0.1,temp_c=0.1):
#         (self,datasets, batch_size, alpha_0, alpha_1, alpha_2, local_epochs, global_iters, dataset_size, seed, temp_a, temp_c, decay_rate):
        self.global_weights = None
#         self.mainQN=Qnetwork(input_size=input_size, num_actions=num_actions)
#         self.targetQN = Qnetwork(input_size=input_size, num_actions=num_actions)
        
        self.graph = graph
        with self.graph.as_default():
            self.mainQN = Qnetwork(self.graph, scope=f'{scope}_main')
            self.targetQN = Qnetwork(self.graph, scope=f'{scope}_target')
            
#         self.users = []
        self.total_train_samples = 0
#         self.dataset_size = dataset_size
        self.local_epochs = local_epochs
        self.global_iter = global_iters
        self.temp_a = temp_a
        self.temp_c = temp_c
        
    # can add para set
    # create nodes here

    def get_global_weights(self):
        return self.global_weights

    def set_global_weights(self, local_weights_list):
        self.global_weights = local_weights_list
        
    def eval_server_policy(self):
        # not finish yet
        # in gym , but here I only have offline dataset ....
        return 0
    
    def server_to_local(self,node,sess_node,sess_fed):
        node.mainQN.set_weights(sess_node,self.mainQN.get_weights(sess_fed))
        node.targetQN.set_weights(sess_node,self.targetQN.get_weights(sess_fed))
        
    def aggregate(self,nodes,sessions,sess_fed):
        print("server:start doing aggregation....")
        # set server paras to zero
        # ....
        # if every aggregate has to set server's para to zero, then self.mainQN.get_weights(sess_fed) is always 0;
        total_train_main = 0
        total_train_target = 0
        # calculate total weights
        for node in nodes:#计算总权重:
            total_train_main += np.exp(self.temp_a * node.pol_val)### not define yet
            total_train_target += np.exp(self.temp_c * node.pol_val)### not define yet
            
#         mainQN_new_weight=None
#         targetQN_new_weight=None  #[]
        # 初始化新的权重为零数组
        node_weights_sample = nodes[0].mainQN.get_weights(sessions[0][0])
        mainQN_new_weight = [np.zeros_like(weight) for weight in node_weights_sample]
        node_weights_sample = nodes[0].targetQN.get_weights(sessions[0][0])
        targetQN_new_weight = [np.zeros_like(weight) for weight in node_weights_sample]
        
        for node, (sess_node, _, _) in zip(nodes, sessions):
            ratio_main = np.exp(self.temp_a * node.pol_val) / total_train_main
            ratio_target = np.exp(self.temp_c * node.pol_val) / total_train_target

            node_mainQN_weights = node.mainQN.get_weights(sess_node)
            node_targetQN_weights = node.targetQN.get_weights(sess_node)

            mainQN_new_weight = [nw + w * ratio_main for nw, w in zip(mainQN_new_weight, node_mainQN_weights)]
            targetQN_new_weight = [nw + w * ratio_target for nw, w in zip(targetQN_new_weight, node_targetQN_weights)]

#     self.mainQN.set_weights(sess_fed, mainQN_new_weight)
        
#         for node, (sess_node, _, _) in zip(nodes, sessions):#计算归一化权重并聚合参数:
#             ratio_main = np.exp(self.temp_a * node.pol_val) / total_train_main
#             ratio_target = np.exp(self.temp_c * node.pol_val) / total_train_target
#             mainQN_new_weight+= node.mainQN.get_weights(sess_node) * ratio_main
#             targetQN_new_weight+= node.targetQN.get_weights(sess_node) * ratio_target
            

        self.mainQN.set_weights(sess_fed,mainQN_new_weight)
        self.targetQN.set_weights(sess_fed,targetQN_new_weight)
    
def Local_to_server(nodes, server):
    local_weights_list = []
    for node in nodes:
        local_weights_list.append(node.model_stats['weights'][-1])
    server.set_global_weights(local_weights_list)


    

#     def aggregate_weights(self, local_weights_list):
#         # Perform Federated Averaging
#         new_weights = []
#         for weights in zip(*local_weights_list):
#             new_weights.append(np.mean(weights, axis=0))
#         self.global_weights = new_weights


aggregated_counts = {}
for node in nodes:
    for state_action, count in node.state_action_counts.items():
        if state_action not in aggregated_counts:
            aggregated_counts[state_action] = 0
        aggregated_counts[state_action] += count
        

In [46]:
def plot_metrics(node):
    plt.figure(figsize=(15, 5))

    # Plot average Q-values from the DQN
    plt.subplot(1, 3, 1)
    plt.plot(node.model_stats['mean_q'], label='Average Q(DQN)')
    plt.xlabel('Training Steps')
    plt.ylabel('Q-value')
    plt.title('Average Q(DQN) over Training Steps')
    plt.legend()

    # Plot Q-values from the physician's policy
    plt.subplot(1, 3, 2)
    plt.plot(node.model_stats['phys_q'], label='Q(Physician)')
    plt.xlabel('Training Steps')
    plt.ylabel('Q-value')
    plt.title('Q(Physician) over Training Steps')
    plt.legend()

    # Plot loss
    plt.subplot(1, 3, 3)
    plt.plot(node.model_stats['losses'], label='Loss')
    plt.xlabel('Training Steps')
    plt.ylabel('Loss')
    plt.title('Loss over Training Steps')
    plt.legend()

    plt.tight_layout()
    plt.show()


## new

In [47]:
def Local_updates_new29(round,sess,saver,node,local_epochs,batch_size,per_alpha = 0.6,per_epsilon = 0.01,gamma = 0.99,tau = 0.001 ):
#     if i donot use summary writer:
    with sess.as_default():
        with node.graph.as_default():
            load_model = True  # Whether to load a saved model.
            save_dir = f"FedDQN_64_2k/dqn_auto_{node.name}_{round}"
            save_path = f"FedDQN_64_2k/dqn_auto_{node.name}_{round}/ckpt"  # The path to save our model to.

            if not os.path.exists(save_dir):
                os.makedirs(save_dir)

            init = tf.global_variables_initializer()
          
                        # 创建摘要操作并确保唯一标签
#             summary_writer = tf.summary.FileWriter(save_dir, sess.graph)
#             with tf.name_scope(f'Loss_Scope_{node.name}'):
#                 loss_summary = tf.summary.scalar(f'Loss_{node.name}', node.mainQN.loss)
#             merged_summary = tf.summary.merge_all()
            ###############
            if load_model:
                print('Trying to load model...')
                try:
                    restorer = tf.train.import_meta_graph(save_path + '.meta')
                    restorer.restore(sess, tf.train.latest_checkpoint(save_dir))
                    print("Model restored")
                except IOError:
                    print("No previous model found, running default init")
                    sess.run(init)
                try:
                    per_weights = pickle.load(open(save_dir + "/per_weights.p", "rb"))
                    imp_weights = pickle.load(open(save_dir + "/imp_weights.p", "rb"))

                    # Restore PER weights
                    node.df['prob'] = per_weights
                    node.df['imp_weight'] = imp_weights
                    print("PER and Importance weights restored")
                except IOError:
                    print("No PER weights found - default being used for PER and importance sampling")
            else:
                print("Running default init")
                sess.run(init)
            print("Init done")
            ###############
            replay_buffer_size=node.train_auto.shape[0]
            print("----node {} replay_buffer_size is {}----".format(node.name,node.train_auto.shape[0]))
#             num_steps = (replay_buffer_size // batch_size)  * local_epochs ####此处修改了！
            num_steps=2000
            print(f"----num_steps is{num_steps}----")

            ################## local training ###################
            print("start local training ...")
            for i in range(num_steps):
                if save_results:
                    do_save_results()
                    break
                net_loss = 0.0
                net_q = 0.0
                states, actions, rewards, next_states, done_flags, sampled_df = process_batch(node, batch_size)

                actions_from_q1 = sess.run(node.mainQN.predict, feed_dict={node.mainQN.state: next_states, node.mainQN.phase: 1})
                cur_act = sess.run(node.mainQN.predict, feed_dict={node.mainQN.state: states, node.mainQN.phase: 1})
                Q2 = sess.run(node.targetQN.q_output, feed_dict={node.targetQN.state: next_states, node.targetQN.phase: 1})
                end_multiplier = 1 - done_flags
                double_q_value = Q2[range(batch_size), actions_from_q1]
                double_q_value[double_q_value > REWARD_THRESHOLD] = REWARD_THRESHOLD
                double_q_value[double_q_value < -REWARD_THRESHOLD] = -REWARD_THRESHOLD
                targetQ = rewards + (gamma * double_q_value * end_multiplier)

                imp_sampling_weights = np.array(sampled_df['imp_weight'] / float(max(node.df['imp_weight'])))
                imp_sampling_weights[np.isnan(imp_sampling_weights)] = 1
                imp_sampling_weights[imp_sampling_weights <= 0.001] = 0.001

                _, loss, error = sess.run([node.mainQN.update_model, node.mainQN.loss, node.mainQN.abs_error],
                                          feed_dict={node.mainQN.state: states,
                                                     node.mainQN.targetQ: targetQ,
                                                     node.mainQN.actions: actions,
                                                     node.mainQN.phase: True,
                                                     node.mainQN.imp_weights: imp_sampling_weights})


                # Write summary to TensorBoard
#                 summary_writer.add_summary(summary, i) ####new
                update_target(target_ops, sess)
                net_loss += sum(error)
                net_q += np.mean(targetQ)
                
                ############ update imp_weight ###########
                new_weights = pow((error + per_epsilon), per_alpha)
                node.df.loc[node.df.index.isin(sampled_df.index), 'prob'] = new_weights
                temp = 1.0 / new_weights
                node.df.loc[node.df.index.isin(sampled_df.index), 'imp_weight'] = pow(((1.0 / len(node.df)) * temp), beta_start)

                if i % 100 == 0 and i > 0:
                    print('------current steps{}/{}-----'.format(i,num_steps))
                    saver.save(sess, save_path)
                    print(f"Saved Model for {node.name}, step is " + str(i))

                    av_loss = net_loss / (100.0*batch_size)
                    print("Average loss is ", av_loss)
                    net_loss = 0.0

                    print("Saving PER and importance weights")
                    with open(save_dir + f'{node.name}_per_weights.p', 'wb') as f:
                        pickle.dump(node.df['prob'], f)
                    with open(save_dir + f'{node.name}_imp_weights.p', 'wb') as f:
                        pickle.dump(node.df['imp_weight'], f)

               
                    print("physactions ", actions)
                    print(" chosen actions ", cur_act)
                    if i % 1000==0:
                        phys_q, phys_actions, agent_q, agent_actions, mean_abs_error = do_eval_new(sess,node,eval_type='val')        
                        print("mean_abs_error:",mean_abs_error)
                        print("phys_q:",np.mean(phys_q))
                        print("agent_q:",np.mean(agent_q))
                        parameters = {
                        "mean_abs_error": mean_abs_error,
                        "mean_phys_q": np.mean(phys_q),
                        "mean_agent_q": np.mean(agent_q)}
                        directory_path = f'{save_dir}/parameters'
                        file_path = f'{directory_path}/parameters{i}.pkl'
                        os.makedirs(directory_path, exist_ok=True)
                        with open(file_path, 'wb') as f:
                            pickle.dump(parameters, f)   
                        node.model_stats['agent_q'].append(np.mean(agent_q))
                        node.model_stats['phys_q'].append(np.mean(phys_q))
                
                node.model_stats['weights'][i] = node.mainQN.get_weights(sess)
                node.model_stats['losses'].append(loss)
                node.model_stats['mean_targetQ'].append(net_q)
                node.model_stats['errors'].append(error)
                
                if i==num_steps-1:
                    save_every(sess,node,round)
                    directory_path = f'FedDQN_64_2k/model_stat/{node.name}'
                    file_path = f'{directory_path}/model_stats_round_{round}_local_iter_{i}.pkl'

                    # Ensure the directory exists
                    os.makedirs(directory_path, exist_ok=True)
                    # Open the file and write the data
                    with open(file_path, 'wb') as f:
                        pickle.dump(node.model_stats, f)
                
                
                # Close the summary writer
#             summary_writer.close()

### main:0612,local iter=5k, wo encoder

In [48]:
df_orig = pd.read_csv('/rds/general/user/xy823/home/env0418/ai_clinician_yxr/py_ai_clinician-master/MIMICtable_li.csv')

In [49]:
len(df_orig)

984010

# main:0617,batch_size=64,global_iters=4, wo encoder,num_steps=10000

In [ ]:
#在主函数中为每个节点创建独立的会话，并在其图上下文中创建 Saver 
if __name__ == "__main__":
    ######### 0. set parameters
    num_nodes = 3  # Example number of nodes
    input_size=len(state_features)
    num_actions=25
    local_epochs=20# 10
    batch_size=64 ##128 ###128
    global_iters=20  #7### 100
#     num_steps = 2000
    beta_start=0.9
    sync_interval = 10  # Define when to synchronize
    counter = 0
    REWARD_THRESHOLD =15
    reg_lambda = 5
    action_map = {}
    count = 0
    save_res=f"FedDQN_64_2k/res"
    for iv in range(5):
        for vaso in range(5):
            action_map[(iv,vaso)] = count
            count += 1

    per_flag=True

    print(state_features)
    
    config = tf.ConfigProto()
    config.gpu_options.allow_growth = True  # Don't use all GPUs 
    config.allow_soft_placement = True  # Enable manual control
    
    tf.reset_default_graph()
    per_alpha = 0.6
    per_epsilon = 0.01
    gamma = 0.99
    tau = 0.001
    load_model=False
    
    ###### 1. create nodes
#     env0418/ai_clinician_yxr/py_ai_clinician-master/continuous/0607/data/nodes/N1_train.csv
    orig_path = "../continuous/data/orig.csv"
    path='../local/data'
    nodes=create_nodes(orig_path,path)
    
#     state_features = [str(i) for i in range(200)]
    print(state_features)
    print(len(state_features))
  
    
    ####### 2. create session graph for each node
    sessions = []
    for node in nodes:
        graph = node.mainQN.fc_1.graph  ###???
        sess = tf.Session(graph=graph, config=config)### # 使用节点的图创建会话
        with sess.as_default():
            with graph.as_default():
                saver = tf.train.Saver(tf.global_variables())
                init = tf.global_variables_initializer()
                sess.run(init)
                print(f"Init done for {node.name}")
                sessions.append((sess, saver, node))
                
    trainables = tf.trainable_variables()
    target_ops = update_target_graph(trainables, tau)
    
    ####### 3. create server  
    # 创建服务器
    graph_server = tf.Graph()
    scope_server = "scope_server"
    with graph_server.as_default():
        server = Server(input_size, num_actions, local_epochs, global_iters, graph_server, scope_server)
        sess_server = tf.Session(graph=graph_server, config=config)
        with sess_server.as_default():
            saver_server = tf.train.Saver(tf.global_variables())
            init = tf.global_variables_initializer()
            sess_server.run(init)
            print("Server init done")
    
    av_q_list = []
    save_results = False
    
    #########4. start main loop ####################
    for glob_iter in range(server.global_iter):
        avg_rwd = []
        print("-------------Round number: ", glob_iter, " -------------")
        for sess, saver, node in sessions:
            with sess.as_default():
#                 with sess.graph.as_default(): # new 
                    server.server_to_local(node,sess,sess_server)
                    global_reward = server.eval_server_policy()
                    print("Local_updates of:",node.name)
#                     print(node.mainQN.update_ops)
                    Local_updates_new29(glob_iter,sess,saver,node,local_epochs,batch_size) ####!!!!!暂时把num——steps=2100
                    server_phy_q,_,server_q,_,_= do_eval_new(sess_server,node,server=server,eval_type='fed')# evaluate fed policy on local dataset
                    node_phy_q,phys_actions,node_q,agent_actions,_= do_eval_new(sess,node,eval_type='test') # evaluate current policy on local dataset
                    print("!!!!check agent_actions shape in test:",len(agent_actions))
                    node.pol_val = np.mean(node_q)
                    node.server_val=np.mean(server_q)
                    print(f"in node{node.name}, the server policy evaluation:  server_phy_q is : {np.mean(server_phy_q)} VS  server_q is {node.server_val} ")
                    print(f"in node{node.name}, the local AI policy evaluation:  phy_q is : {np.mean(node_phy_q)} VS  AI_q is {node.pol_val} ")
            #     # 保存当前的actor+
                        # Save variables to a pickle file
                    save_data = {
                        'node_phy_q': node_phy_q,
                        'phys_actions': phys_actions,
                        'node_q': node_q,
                        'agent_actions': agent_actions
                    }
                    with open(f'{save_res}/{node.name}_round_{glob_iter}.pkl', 'wb') as f:
                        pickle.dump(save_data, f)
            #     node.prev_actor = copy.deepcopy(node.actor) ###
                    if node.server_val > node.pol_val:
                        print(" node.server_val > node.pol_val: decaying current node.")
                        node.decay = node.decay * node.decay_rate      ##暂无作用 # node.decay,self.decay加在哪里需要考虑
    #                 node_reward=node.eval_policy()# in gym
                    node_reward=node.pol_val  ### which is  np.mean(node_q)
                    avg_rwd.append(node_reward)
                    
        print('Average reward over clients:', np.mean(avg_rwd))
#         Local_to_server(nodes, server)
        server.aggregate(nodes,sessions,sess_server)
        with open(f'{save_res}/final_reward_3_nodes_round_{glob_iter}.pkl', 'wb') as f:
            pickle.dump(avg_rwd, f)


['Albumin', 'Arterial_BE', 'Arterial_lactate', 'Arterial_pH', 'BUN', 'CO2_mEqL', 'Calcium', 'Chloride', 'Creatinine', 'DiaBP', 'FiO2_1', 'GCS', 'Glucose', 'HCO3', 'HR', 'Hb', 'INR', 'Ionised_Ca', 'Magnesium', 'MeanBP', 'PT', 'PTT', 'PaO2_FiO2', 'Platelets_count', 'Potassium', 'RR', 'SGOT', 'SGPT', 'SIRS', 'SOFA', 'Shock_Index', 'Sodium', 'SpO2', 'SysBP', 'Temp_C', 'Total_bili', 'WBC_count', 'Weight_kg', 'age', 'elixhauser', 'gender', 'mechvent', 'output_4hourly', 'output_total', 'paCO2', 'paO2', 're_admission', 'bloc']
(87024, 85)
(329352, 85)
(327320, 85)
['Albumin', 'Arterial_BE', 'Arterial_lactate', 'Arterial_pH', 'BUN', 'CO2_mEqL', 'Calcium', 'Chloride', 'Creatinine', 'DiaBP', 'FiO2_1', 'GCS', 'Glucose', 'HCO3', 'HR', 'Hb', 'INR', 'Ionised_Ca', 'Magnesium', 'MeanBP', 'PT', 'PTT', 'PaO2_FiO2', 'Platelets_count', 'Potassium', 'RR', 'SGOT', 'SGPT', 'SIRS', 'SOFA', 'Shock_Index', 'Sodium', 'SpO2', 'SysBP', 'Temp_C', 'Total_bili', 'WBC_count', 'Weight_kg', 'age', 'elixhauser', 'gender',

In [ ]:
# nodes[0].model_stats

In [ ]:
# nodes[0].model_stats

In [ ]:
# nodes[0].__dict__

In [ ]:
nodes[0].name